# 01 — Data Cleaning

**Inputs needed:** `data/raw/<PID>/before/` DICOMs and a `configs/default.yaml` with `preprocessing.target_phase`.
**Outputs produced:** `data/processed/<PID>/before*.nii.gz` and `crop_metadata.json`.
**Runtime:** ~1–3 minutes per patient on GPU (TotalSegmentator dominates).


End-to-end Phase 2 walkthrough for one or more patients:

1. **Phase filtering + DICOM → NIfTI** via `DICOMLoader`.
2. **HU windowing + Z-score (within liver mask)** via `preprocess_volume`.
3. **Liver segmentation** with TotalSegmentator via `LiverSegmentor`.
4. **Bounding-box cropping + crop_metadata.json** via `crop_patient`.

Produces `data/processed/<PID>/{before,before_liver,before_cropped}.nii.gz` and the matching `crop_metadata.json`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.utils.config import ensure_dirs, load_config, set_seed
from src.utils.logger import setup_logger

cfg = load_config(ROOT / "configs" / "default.yaml")
ensure_dirs(cfg)
set_seed(int(cfg.get("seed", 42)))
setup_logger("hcc", log_file=Path(cfg["paths"]["logs_dir"]) / "preprocessing.log")
cfg["paths"]

In [ ]:
from src.data.dicom_loader import DICOMLoader
from src.data.liver_segmentation import LiverSegmentor
from src.data.preprocessing import preprocess_volume
from src.data.cropping import crop_patient

raw_dir = Path(cfg["paths"]["raw_dir"])
processed_dir = Path(cfg["paths"]["processed_dir"])

loader = DICOMLoader(raw_dir, cfg["preprocessing"]["target_phase"], processed_dir)
segmentor = LiverSegmentor(
    gpu=bool(cfg["preprocessing"]["liver_segmentation"].get("gpu", True))
)

patient_ids = sorted(p.name for p in raw_dir.iterdir() if p.is_dir())
print(f"Patients found in {raw_dir}: {patient_ids[:10]} (total={len(patient_ids)})")

## Run the pipeline for a single patient (smoke test)

Pick the first patient and run the full chain. Re-run the loop in the next cell to process them all.

In [ ]:
if patient_ids:
    pid = patient_ids[0]
    print(f"Processing {pid}")
    volume_path = loader.convert_to_nifti(pid)
    mask_path = segmentor.segment(volume_path)
    _, stats = preprocess_volume(volume_path, mask_path, cfg)
    cropped_path, meta_path, cropped_mask_path = crop_patient(
        volume_path.parent,
        zscore_stats=stats,
    )
    print(
        {
            "volume": str(volume_path),
            "mask": str(mask_path),
            "cropped": str(cropped_path),
            "cropped_mask": str(cropped_mask_path),
            "meta": str(meta_path),
        }
    )

In [ ]:
from tqdm.notebook import tqdm

errors = {}
for pid in tqdm(patient_ids, desc="Preprocessing"):
    try:
        volume_path = loader.convert_to_nifti(pid)
        mask_path = segmentor.segment(volume_path)
        _, stats = preprocess_volume(volume_path, mask_path, cfg)
        _, _, _ = crop_patient(volume_path.parent, zscore_stats=stats)
    except Exception as exc:
        errors[pid] = str(exc)
errors

## Sanity-check one cropped output


In [ ]:
import json
import nibabel as nib
import numpy as np

if patient_ids:
    pid = patient_ids[0]
    pdir = processed_dir / pid
    vol = nib.load(str(pdir / "before_cropped.nii.gz")).get_fdata()
    meta = json.loads((pdir / "crop_metadata.json").read_text())
    print("Cropped shape :", vol.shape)
    print("BBox          :", meta["bbox"])
    print("Z-score stats :", meta.get("zscore"))
    print("Voxel mean/std:", float(np.mean(vol)), float(np.std(vol)))